# 01.1 — Objects and state

[Class index and setup](README.md) · [Next: 01.2](01.2-behavior-and-composition.ipynb)

**Purpose:** Give a small collection a clear owner and explain how changing one object affects it.

**In class:** 30 minutes. **Independent work:** 15 minutes, included in the class index's 210-minute allowance. The separately assessed project adds workload.

## What you will be able to do

- Create instances and inspect their attributes.
- Explain what the constructor and self do.
- Keep mutable state separate for each instance.

## Starting point

Know Python functions, lists, and dictionaries. A class definition will be new; the setup instructions are in the index.

This file includes all definitions and inputs it needs. Read the parts in order, but start this one with **Restart Kernel → Run All** in VS Code. Do not execute another notebook or an earlier exercise solution to prepare it. Empty answer cells are safe during Run All.

Reference solutions stay in separate cells: attempt the task, uncomment its `%load` line, and run that cell to reveal source. Inspect the result before running the populated cell separately. Keep your answer in its own cell because loading can replace the invoking cell. A loaded solution remains in the notebook even after restarting the kernel.

In [ ]:
from pathlib import Path
import sys

if not Path("solutions/exercise_01_1_e1.py").is_file():
    raise FileNotFoundError(
        "Open 01-oop-and-schemas.code-workspace in VS Code and restart the kernel. "
        "The working directory must be the class folder containing README.md and solutions/."
    )
print("Notebook kernel:", sys.executable)
print("Class files are available.")

## In class · 0–20 minutes — From a record to a report

A record can be a dictionary: `{"label": "north", "value": 2.5}`. A list holds several records. For a one-off calculation, a function taking that list can be enough. When a program repeatedly names, adds to, and summarizes a collection, a small object can give that state and behavior a clear home.

Our observations are **nonnegative quantities**, such as counted amounts. The same constraint would not suit every kind of measurement: temperature, for example, can be negative. For this first part we use trusted, local dictionaries. A class by itself does not make arbitrary incoming data valid; part 01.3 will add that boundary.

### Read this definition from the outside in

- `class Report:` defines a type of object.
- Calling `Report("daily")` constructs an **instance**.
- `__init__` initializes a newly created instance. It is not where we download data or perform lengthy analysis.
- `self` refers to the particular instance receiving the operation. Python supplies it for `report.add(...)`.
- `self.name` and `self.records` are **attributes** storing that instance's state.
- A function defined on the class, such as `add`, is used as a **method**.

Run the definition. Defining the class does not itself create a report.

In [ ]:
class Report:
    """A named collection of trusted, local measurement dictionaries."""

    def __init__(self, name):
        self.name = name
        self.records = []

    def add(self, record):
        self.records.append(record)

    def total(self):
        return sum(record["value"] for record in self.records)

### Illustration — CPT-OOP-objects and classes

<img src="images/objects-and-classes.png" alt="One car class is shown beside three named example objects: polo, mini, and beetle." width="500" style="max-width: 100%; height: auto; background-color: white;">

The class defines a common structure; each constructed object has its own identity and state. Treat the cars as a simple analogy, then return to two Report instances and compare their separate records lists. The picture does not establish how Python stores those lists.

*Credit: [CPT-OOP-objects and classes](https://commons.wikimedia.org/wiki/File:CPT-OOP-objects_and_classes.svg) by Pluke, [CC0 1.0 Universal dedication](https://creativecommons.org/publicdomain/zero/1.0/). Commons PNG rendering; no content changes. [Image sources and teaching notes](IMAGE_SUGGESTIONS.md).*

### Construct, change, observe

Predict the total before running the cell. Construction gives the report an empty list. Calling `add` changes that list; calling `total` computes a result from the current contents. The caller does not pass `self` explicitly.

In [ ]:
daily = Report("daily")
print("Initially:", daily.name, daily.records, daily.total())
daily.add({"label": "north", "value": 2.5})
daily.add({"label": "south", "value": 3.5})
print("After adding:", daily.records)
print("Total:", daily.total())
assert daily.total() == 6

The total is `6.0`. An `assert` is a small executable check: it raises an error if the claimed behavior is false. Here it checks the demonstration, not an unfinished student answer. We will study testing more systematically in class 03.

### A class is a definition; instances have their own identity

Two calls to the constructor create two objects. Two initially empty lists can be equal in value without being the **same** list. `==` compares their values; `is` checks identity. Predict all three results below.

In [ ]:
first = Report("first")
second = Report("second")
print("Same report object?", first is second)
print("Equal list values?", first.records == second.records)
print("Same list object?", first.records is second.records)
assert first is not second and first.records is not second.records

The results are `False`, `True`, `False`. The constructor's `self.records = []` is executed separately for each instance. That is the property the next exercise asks you to demonstrate.

### 🧩 Exercise 01.1.E1 — Independent instance state

> **In class · 10 minutes**
>
> **Goal:** Connect a visible change to one instance-owned list.
>
> **Start here:** The Report definition in this file and the answer cell below.
>
> **Task:** Create reports named left and right. Add a new record with value 4 to left only. Print both totals and check that the lists are different objects. Explain where each list was created.
>
> **Check:** The totals are 4 and 0; adding to left leaves right unchanged.
>
> **Hint:** Call the constructor twice. Use `left.records is not right.records`; do not assign right = left.

In [ ]:
# Your attempt for 01.1.E1.
# Keep your code and explanation here, separate from the reference cell.

In [ ]:
# Reveal only after attempting the exercise.
# Remove the leading '# ' from the next line and run this cell.
# %load solutions/exercise_01_1_e1.py

## Guided follow-up — Independent work · 15 minutes

### Three common mistakes

**Confusing another name with another instance.** `alias = daily` does not construct anything. Both names refer to the same object. An operation through either name is visible through the other.

In [ ]:
original = Report("original")
alias = original
alias.add({"label": "east", "value": 1})
print("Same object:", original is alias, "Total:", original.total())
assert original is alias and original.total() == 1

**Putting mutable state on the class.** This is an intentionally wrong sketch; do not replace the working definition with it:

```python
class SharedReport:
    records = []  # one class-level list, shared by instances using this attribute
```

Creating a list in `__init__` gives each instance its own list. A mutable default argument such as `def __init__(self, records=[])` can also share a list, because defaults are evaluated when the function is defined. For now, initialize the empty list inside the constructor.

**Assuming separate lists imply deep copies of their contents.** Our `add` method stores the dictionary object it receives. If two reports deliberately store the same dictionary, changing that dictionary can affect both views. This lesson's exercise creates one record and changes only the owning list; it does not claim to deep-copy arbitrary objects.

### When is the class useful?

A report gives a name and repeated operations to accumulated state. If all you need is `sum(row["value"] for row in rows)` once, a function or expression can be clearer. Creating a class for every dictionary is not a learning goal. Be able to say which state and operations make this class worthwhile.

### Completion check and reflection

Within these 15 minutes, explain the difference between a class, an instance, and another variable referring to that instance. Predict what happens to `alias` when `original` changes. Describe the shared-list mistake and how the constructor avoids it.

**Optional extension · 10 minutes, excluded from required time:** demonstrate sharing one input dictionary between two reports, then explain what is and is not independent.

## References

- [Python classes](https://docs.python.org/3/tutorial/classes.html)
- [Next: behavior and composition](01.2-behavior-and-composition.ipynb)
- [Class index](README.md)